# Project 4: Reinforcement Learning for Print Speed Control  
## Train a Q-Learning Agent Using Real 3D Printer Thermal Data

In this project, we will train a simple reinforcement learning agent to control **print speed** during a simulated 3D printing process.

The goal is:

> Choose the print speed layer by layer so that the measured interface temperature stays inside a target temperature window.

We will use real fixed-speed Prusa printer data that has already been prepared for you. The data comes from six prints performed at different fixed speeds.

By the end of this notebook, you will create a learned Q-table that tells the agent what speed action to take for different temperature and speed conditions.


# This notebook has selected code blocks left incomplete. Look for `TODO` comments and `_____` blanks. #

Your goal is to complete the missing parts, run the notebook from top to bottom, and explain what the learned Q-table is doing.

Suggested strategy:

1. Read the markdown before each code cell.
2. Fill only one cell at a time.
3. Run the cell and check the output before moving forward.
4. Use the hints in the comments, not the completed notebook.


## What are we trying to learn?

The printer can print at different speeds. The speed affects the measured temperature:

- slower speed usually gives a warmer measured region
- faster speed usually gives a cooler measured region
- a middle speed may keep the temperature closer to the target window

The reinforcement learning agent will learn this behavior from data.

At each layer, the agent observes:

```text
current temperature range + current print speed
```

Then it chooses one action:

```text
slow down, hold speed, or speed up
```

After the action, the simulator gives the next temperature and a reward.


## Project workflow

We will follow these steps:

1. Load the prepared real printer data.
2. Define the RL states, actions, and reward.
3. Build a simple simulator from the real transition data.
4. Train a Q-learning table.
5. Visualize the learned Q-table and policy.
6. Test the learned policy in the simulator.
7. Save the final Q-table.

The final output of this notebook is:

```text
q_learning_real_outputs/q_table.npy
```

This file contains the trained Q-table.


In [1]:
# ============================================================
# Cell 1: Import libraries
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1: Load the processed data files

Before running this notebook, make sure the prepared Project 4 data folder is available.

The folder should be named:

```text
project4_real_data_outputs/
```

It should contain these three CSV files:

```text
project4_real_fixed_speed_combined.csv
project4_real_transition_samples.csv
project4_real_speed_summary.csv
```

The most important file is:

```text
project4_real_transition_samples.csv
```

This file contains layer-to-layer temperature transitions. We will use it to build the simulator for Q-learning.


In [7]:
# ============================================================
# Cell 2: Set data folder path
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# TODO 1:
# Set PROJECT_FOLDER to the folder that contains project4_real_data_outputs.
# Example format:
PROJECT_FOLDER = Path("/content/drive/MyDrive/Colab_Notebooks/SUPREME_2026/Project_04")

DATA_DIR = PROJECT_FOLDER / "project4_real_data_outputs"

combined_path = DATA_DIR / "project4_real_fixed_speed_combined.csv"
transition_path = DATA_DIR / "project4_real_transition_samples.csv"
summary_path = DATA_DIR / "project4_real_speed_summary.csv"

print("Data folder:", DATA_DIR.resolve())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data folder: /content/drive/MyDrive/Colab_Notebooks/SUPREME_2026/Project_04/project4_real_data_outputs


In [8]:
# ============================================================
# Cell 3: Load the data files
# ============================================================

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data folder not found: {DATA_DIR}\n"
        "Check PROJECT_FOLDER and make sure project4_real_data_outputs is inside it."
    )

required_files = [combined_path, transition_path, summary_path]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing file: {file_path}\n"
            "Please make sure all three required CSV files are inside project4_real_data_outputs."
        )

df = pd.read_csv(combined_path)
transitions = pd.read_csv(transition_path)
speed_summary = pd.read_csv(summary_path)

print("Files loaded successfully.")
print("Combined data shape:", df.shape)
print("Transition data shape:", transitions.shape)
print("Speed summary shape:", speed_summary.shape)

transitions.head()


Files loaded successfully.
Combined data shape: (300, 19)
Transition data shape: (294, 10)
Speed summary shape: (6, 10)


,source_file,print_speed_mms,layer,temp_now_C,temp_next_C,delta_T_C,state_now,state_next_if_hold,temp_bin_now,temp_bin_next
0,print_001_100mms_20260611_104506.csv,100.0,1,41.16,40.84,-0.32,15,15,2,2
1,print_001_100mms_20260611_104506.csv,100.0,2,40.84,40.75,-0.09,15,15,2,2
2,print_001_100mms_20260611_104506.csv,100.0,3,40.75,40.56,-0.19,15,15,2,2
3,print_001_100mms_20260611_104506.csv,100.0,4,40.56,40.34,-0.22,15,15,2,2
4,print_001_100mms_20260611_104506.csv,100.0,5,40.34,40.84,0.50,15,15,2,2


## What is in the data?

The combined data file contains the real printer measurements from the six fixed-speed prints.

The transition file contains rows like this:

```text
current layer temperature → next layer temperature
```

That is exactly what we need for reinforcement learning because Q-learning needs to know what happens after an action.

The transition file includes:

- current print speed
- current temperature bin
- current state
- next state if the same speed is held
- temperature change from one layer to the next


In [ ]:
# ============================================================
# Cell 4: Quick data check
# ============================================================

print("Print speeds in the dataset:")
print(sorted(df["print_speed_mms"].unique()))

print("\nObserved ROI max temperature range:")
print("Minimum:", round(df["roi_max"].min(), 2), "°C")
print("Maximum:", round(df["roi_max"].max(), 2), "°C")

print("\nTransition data columns:")
print(transitions.columns.tolist())

speed_summary

## Step 2: Define the RL environment

Now we define the reinforcement learning problem.

### Observation

The agent observes:

```text
temperature bin + current speed
```

### Action

The agent can choose:

| Action number | Meaning |
|---|---|
| 0 | slow down |
| 1 | hold speed |
| 2 | speed up |

### Reward

The agent receives:

- positive reward if the next temperature is inside the target window
- negative reward if the next temperature is outside the target window


## Target temperature window

The observed ROI max temperature in this dataset is approximately:

```text
31.8°C to 45.9°C
```

For this classroom exercise, we use the following target window:

```text
40.0°C to 43.5°C
```

This is the temperature range that we want the agent to maintain in the simulator.


In [ ]:
# ============================================================
# Cell 5: Define speeds, target window, states, and actions
# ============================================================

SPEEDS = np.array(sorted(df["print_speed_mms"].dropna().unique()), dtype=float)

# TODO 3:
# Fill in the lower and upper limits of the target temperature window.
# Hint: use the target window described in the markdown above this cell. (40.0 and 43.5)

TARGET_LO = _____
TARGET_HI = _____

TEMP_BIN_NAMES = [
    "far cold",
    "cold",
    "in target",
    "hot",
    "far hot",
]

# TODO 4:
# Complete the action dictionary.
# Hint: action 0 decreases speed, action 1 keeps speed the same, action 2 increases speed. For example- 0: "slow down", 1: "hold speed", 2: "speed up"

ACTION_NAMES = {
    0: "_____",
    1: "_____",
    2: "_____",
}

N_TEMP_BINS = len(TEMP_BIN_NAMES)
N_SPEEDS = len(SPEEDS)
N_STATES = N_TEMP_BINS * N_SPEEDS
N_ACTIONS = len(ACTION_NAMES)

print("Observed ROI max range:", round(df["roi_max"].min(), 2), "to", round(df["roi_max"].max(), 2), "°C")
print("Target window:", TARGET_LO, "to", TARGET_HI, "°C")
print("Speed levels:", SPEEDS)
print("Number of temperature bins:", N_TEMP_BINS)
print("Number of states:", N_STATES)
print("Number of actions:", N_ACTIONS)


## Temperature bins

We divide the measured temperature into five simple bins:

| Bin number | Name | Temperature range |
|---|---|---|
| 0 | far cold | T < 36.0°C |
| 1 | cold | 36.0°C ≤ T < 40.0°C |
| 2 | in target | 40.0°C ≤ T ≤ 43.5°C |
| 3 | hot | 43.5°C < T ≤ 45.0°C |
| 4 | far hot | T > 45.0°C |

The state combines the temperature bin and the current speed.

Because we have 5 temperature bins and 6 speed levels:

```text
5 × 6 = 30 states
```


In [ ]:
# ============================================================
# Cell 6: Helper functions for states and rewards
# ============================================================

def temp_to_bin(temp_C):
    """Convert continuous ROI max temperature to a temperature bin index."""
    # TODO 5:
    # Complete the temperature bin logic.
    # Hint:
    # 0 = far cold, 1 = cold, 2 = in target, 3 = hot, 4 = far hot. So in the blank spaces we need to fill in 0, 1, 2, 3 and 4 sequentially.
    if temp_C < 36.0:
        return _____
    elif temp_C < TARGET_LO:
        return _____
    elif temp_C <= TARGET_HI:
        return _____
    elif temp_C <= 45.0:
        return _____
    else:
        return _____


def speed_to_index(speed_mms):
    """Convert speed value into speed index."""
    matches = np.where(SPEEDS == float(speed_mms))[0]
    if len(matches) == 0:
        raise ValueError(f"Speed {speed_mms} is not in SPEEDS = {SPEEDS}")
    return int(matches[0])


def state_from_temp_and_speed(temp_C, speed_mms):
    """Convert temperature and speed into a single state number."""
    temp_bin = temp_to_bin(temp_C)
    speed_idx = speed_to_index(speed_mms)

    # TODO 6:
    # Combine temp_bin and speed_idx into one state number.
    # Hint: each temperature bin contains N_SPEEDS states. We need to fill the blank spaces like this: temp_bin * N_SPEEDS + speed_idx
    return _____ * N_SPEEDS + _____


def describe_state(state):
    """Describe a state in human-readable form."""
    temp_bin = state // N_SPEEDS
    speed_idx = state % N_SPEEDS

    return {
        "state": int(state),
        "temp_bin": int(temp_bin),
        "temp_label": TEMP_BIN_NAMES[temp_bin],
        "speed_index": int(speed_idx),
        "speed_mms": float(SPEEDS[speed_idx]),
    }


def calculate_reward(temp_C):
    """Reward based on the target window."""
    # TODO 7:
    # Return +1.0 inside the target window.
    # Return a negative penalty when the temperature is too cold or too hot.
    # Hint: use TARGET_LO and TARGET_HI.
    if _____ <= temp_C <= _____:      # TARGET_LO <= temp_C <= TARGET_HI:
        return _____                  # return 1.0
    elif temp_C < TARGET_LO:
        return -0.25 * (_____ - temp_C) # return -0.25 * (TARGET_LO - temp_C)
    else:
        return -0.25 * (temp_C - _____)  # return -0.25 * (temp_C - TARGET_HI)


# Quick examples
example_state = state_from_temp_and_speed(42.0, 70.0)

print("Example:")
print("Temperature = 42°C, speed = 70 mm/s")
print("State number:", example_state)
print(describe_state(example_state))

print("Reward examples:")
for temp in [34, 38, 41, 44, 47]:
    print(f"T = {temp:>5.1f} °C  reward = {calculate_reward(temp):>6.2f}")


## Step 3: Build the real-data simulator

The simulator is built from the transition data.

For each row, we know:

```text
current speed
current temperature bin
temperature change to the next layer
```

We store these changes in a lookup table.

Later, when the agent chooses a speed, the simulator samples a realistic temperature change from the matching group.


In [ ]:
# ============================================================
# Cell 7: Build transition lookup dictionary
# ============================================================

transition_lookup = {}

for _, row in transitions.iterrows():
    key = (float(row["print_speed_mms"]), int(row["temp_bin_now"]))
    transition_lookup.setdefault(key, []).append(float(row["delta_T_C"]))

transition_lookup = {
    key: np.array(values, dtype=float)
    for key, values in transition_lookup.items()
}

# Fallback data for each speed.
# If a specific speed-temperature bin is missing, the simulator uses all
# observed temperature changes for that speed.
fallback_by_speed = {}

for speed, group in transitions.groupby("print_speed_mms"):
    fallback_by_speed[float(speed)] = group["delta_T_C"].to_numpy(dtype=float)

print("Number of transition lookup entries:", len(transition_lookup))
print("\nAvailable speed-bin transition groups:")

for key in sorted(transition_lookup.keys()):
    speed, temp_bin = key
    print(f"speed={speed:>6.1f} mm/s, bin={temp_bin} ({TEMP_BIN_NAMES[temp_bin]}), n={len(transition_lookup[key])}")

## What does the transition lookup mean?

The transition lookup tells us where the real data has examples.

For example:

```text
speed = 70 mm/s, bin = in target, n = 48
```

means:

> The dataset contains 48 layer-to-layer transitions where the printer was running at 70 mm/s and the current temperature was inside the target bin.

The full possible coverage is:

```text
6 speeds × 5 temperature bins = 30 possible groups
```

But not all groups appear in the data. That is normal. A very fast print may never become hot, and a slow print may never become far cold.

The simulator is most reliable in the groups with many samples.


In [ ]:
# ============================================================
# Cell 8: Visualize transition coverage
# ============================================================

coverage_grid = np.zeros((N_TEMP_BINS, N_SPEEDS))

for (speed, temp_bin), values in transition_lookup.items():
    speed_idx = speed_to_index(speed)
    coverage_grid[temp_bin, speed_idx] = len(values)

plt.figure(figsize=(8, 5))
plt.imshow(coverage_grid, aspect="auto", cmap="coolwarm")
plt.colorbar(label="Number of real transition samples")

plt.xticks(range(N_SPEEDS), [f"{s:.0f}" for s in SPEEDS])
plt.yticks(range(N_TEMP_BINS), TEMP_BIN_NAMES)

for i in range(N_TEMP_BINS):
    for j in range(N_SPEEDS):
        plt.text(j, i, int(coverage_grid[i, j]), ha="center", va="center")

plt.xlabel("Print speed (mm/s)")
plt.ylabel("Temperature bin")
plt.title("Real Data Transition Coverage")
plt.tight_layout()
plt.show()

## How to read the coverage plot

Each cell shows how many real transition samples exist for a speed and temperature bin.

Large number:

```text
The simulator has strong real-data support here.
```

Zero:

```text
The real prints never visited this condition.
```

For example, if 170 mm/s only appears in the cold bins, that means the 170 mm/s print was generally too cold. If 20 mm/s appears in the hot bins, that means the slow print was warmer.

This plot helps us understand the limits of the data.


In [ ]:
# ============================================================
# Cell 9: Simulator step function
# ============================================================

def simulator_step(current_temp_C, current_speed_index, action, rng=None):
    """
    Simulate one layer transition.

    Inputs:
        current_temp_C      : current ROI max temperature
        current_speed_index : index of current speed in SPEEDS
        action              : 0 slow down, 1 hold, 2 speed up

    Returns:
        next_temp_C
        next_speed_index
        reward
        next_state
    """
    if rng is None:
        rng = np.random.default_rng()

    # Convert action to speed index change:
    # action 0 -> -1
    # action 1 ->  0
    # action 2 -> +1
    speed_change = action - 1

    next_speed_index = int(np.clip(
        current_speed_index + speed_change,
        0,
        N_SPEEDS - 1
    ))

    chosen_speed = float(SPEEDS[next_speed_index])
    current_temp_bin = temp_to_bin(current_temp_C)

    key = (chosen_speed, current_temp_bin)

    if key in transition_lookup:
        possible_delta_T = transition_lookup[key]
    else:
        possible_delta_T = fallback_by_speed[chosen_speed]

    sampled_delta_T = float(rng.choice(possible_delta_T))

    # Small extra noise prevents every simulated episode from being identical.
    noise = float(rng.normal(0, 0.15))

    next_temp_C = current_temp_C + sampled_delta_T + noise

    # Keep the simulated temperature inside a reasonable range.
    next_temp_C = float(np.clip(next_temp_C, 28.0, 50.0))

    reward = calculate_reward(next_temp_C)
    next_state = state_from_temp_and_speed(next_temp_C, SPEEDS[next_speed_index])

    return next_temp_C, next_speed_index, reward, next_state


# Quick simulator test
rng = np.random.default_rng(123)

next_temp, next_speed_idx, reward, next_state = simulator_step(
    current_temp_C=42.0,
    current_speed_index=speed_to_index(70.0),
    action=2,
    rng=rng
)

print("Simulator test:")
print("Next temp:", round(next_temp, 2), "°C")
print("Next speed:", SPEEDS[next_speed_idx], "mm/s")
print("Reward:", round(reward, 2))
print("Next state:", next_state)
print(describe_state(next_state))

## Step 4: Q-learning

Now we train the Q-table.

The Q-table stores the value of each action in each state.

```text
Q-table shape = number of states × number of actions
```

For this project:

```text
30 states × 3 actions
```

Q-learning update rule:

```text
Q(s,a) ← Q(s,a) + α [r + γ max Q(s′,a′) − Q(s,a)]
```

Meaning:

- `s` is the current state
- `a` is the chosen action
- `r` is the reward
- `s′` is the next state
- `α` controls how fast the table updates
- `γ` controls how much future rewards matter


In [ ]:
# ============================================================
# Cell 10: Q-learning hyperparameters
# ============================================================

# TODO 15:
# Choose the Q-learning hyperparameters.
# Suggested values are given in the comments.

ALPHA = _____          # learning rate, suggested: 0.10
GAMMA = _____          # discount factor, suggested: 0.95

EPSILON_START = _____  # high exploration at first, suggested: 0.80
EPSILON_END = _____    # low exploration near the end, suggested: 0.05

N_EPISODES = _____             # suggested: 800
N_LAYERS_PER_EPISODE = _____   # suggested: 50

INITIAL_TEMP_C = 42.0
INITIAL_SPEED_MMS = 70.0

TRAINING_SEED = 100

print("Q-table shape will be:", (N_STATES, N_ACTIONS))


## Exploration and exploitation

During training, the agent uses an epsilon-greedy strategy.

At the beginning, epsilon is high, so the agent explores more random actions.

Near the end, epsilon is low, so the agent mostly uses the best action it has learned.

This helps the agent try different actions before settling on a policy.


In [ ]:
# ============================================================
# Cell 11: Epsilon-greedy action selection
# ============================================================

def epsilon_by_episode(episode, n_episodes=N_EPISODES):
    """Linearly decay epsilon from EPSILON_START to EPSILON_END."""
    fraction = episode / max(n_episodes - 1, 1)
    epsilon = EPSILON_START + fraction * (EPSILON_END - EPSILON_START)
    return float(epsilon)


def choose_action(q_table, state, epsilon, rng):
    """Choose action using epsilon-greedy exploration."""
    if rng.random() < epsilon:
        return int(rng.integers(N_ACTIONS))
    return int(np.argmax(q_table[state]))


for ep in [0, 50, 100, 150, 200, 250, 300, 350, 400, 600, 800]: # Remember that the llast number should be equal or less than the N_EPISODES you have defined in the previous cell otherwise it will not work.
    print(f"Episode {ep:>4}: epsilon = {epsilon_by_episode(ep):.3f}")

## Step 5: Train the Q-table

Each episode represents one simulated 50-layer print.

For each layer, the agent:

1. observes the current state
2. chooses an action
3. receives the next temperature from the simulator
4. receives a reward
5. updates the Q-table

After many episodes, the Q-table should learn which speed action is usually best.


In [ ]:
# ============================================================
# Cell 12: Train Q-learning
# ============================================================

q_table = np.zeros((N_STATES, N_ACTIONS), dtype=float)
rng = np.random.default_rng(TRAINING_SEED)

episode_logs = []
final_episode_rows = []

for episode in range(N_EPISODES):
    epsilon = epsilon_by_episode(episode)

    current_temp = float(INITIAL_TEMP_C + rng.normal(0, 0.8))
    current_speed_index = speed_to_index(INITIAL_SPEED_MMS)

    total_reward = 0.0
    layers_in_target = 0

    for layer in range(1, N_LAYERS_PER_EPISODE + 1):
        state = state_from_temp_and_speed(current_temp, SPEEDS[current_speed_index])
        action = choose_action(q_table, state, epsilon, rng)

        q_before = q_table[state, action]

        next_temp, next_speed_index, reward, next_state = simulator_step(
            current_temp,
            current_speed_index,
            action,
            rng=rng
        )

        # Bellman update
        best_future_q = np.max(q_table[next_state])
        td_target = reward + GAMMA * best_future_q
        td_error = td_target - q_before
        q_table[state, action] = q_before + ALPHA * td_error

        total_reward += reward
        layers_in_target += int(TARGET_LO <= next_temp <= TARGET_HI)

        if episode == N_EPISODES - 1:
            final_episode_rows.append({
                "episode": episode + 1,
                "layer": layer,
                "state": state,
                "temp_C": current_temp,
                "speed_mms": SPEEDS[current_speed_index],
                "action": action,
                "action_name": ACTION_NAMES[action],
                "next_temp_C": next_temp,
                "next_speed_mms": SPEEDS[next_speed_index],
                "reward": reward,
                "next_state": next_state,
                "q_before": q_before,
                "q_after": q_table[state, action],
                "epsilon": epsilon,
            })

        current_temp = next_temp
        current_speed_index = next_speed_index

    episode_logs.append({
        "episode": episode + 1,
        "epsilon": epsilon,
        "total_reward": total_reward,
        "pct_layers_in_target": 100 * layers_in_target / N_LAYERS_PER_EPISODE,
    })

training_history = pd.DataFrame(episode_logs)
final_training_episode = pd.DataFrame(final_episode_rows)

print("Training complete.")
print("Q-table shape:", q_table.shape)
training_history.tail()

## Step 6: Check training performance

We now plot two things:

1. total reward per episode
2. percentage of layers inside the target window

Because the simulator is random, the curves will be noisy. The rolling mean shows the overall trend more clearly.


In [ ]:
# ============================================================
# Cell 13: Plot total reward over training
# ============================================================

window = 50

plt.figure(figsize=(10, 5))

plt.plot(
    training_history["episode"],
    training_history["total_reward"],
    alpha=0.25,
    label="episode reward"
)

plt.plot(
    training_history["episode"],
    training_history["total_reward"].rolling(window).mean(),
    linewidth=2.5,
    label=f"{window}-episode rolling mean"
)

plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Q-learning Training Reward")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Step 7: Inspect the learned Q-table

The Q-table has one row for each state and one column for each action.

For each state, the best action is the action with the largest Q-value.


In [ ]:
# ============================================================
# Cell 14: Q-table as a readable DataFrame
# ============================================================

q_df = pd.DataFrame(q_table, columns=[ACTION_NAMES[a] for a in range(N_ACTIONS)])
state_descriptions = pd.DataFrame([describe_state(s) for s in range(N_STATES)])

q_table_readable = pd.concat([state_descriptions, q_df], axis=1)

q_table_readable

In [ ]:
# ============================================================
# Cell 15: Q-table heatmap
# ============================================================

plt.figure(figsize=(8, 8))

plt.imshow(q_table, aspect="auto", cmap="RdBu")
plt.colorbar(label="Q-value")

plt.xticks(range(N_ACTIONS), [ACTION_NAMES[a] for a in range(N_ACTIONS)])
plt.yticks(range(N_STATES), [f"S{s}" for s in range(N_STATES)])

plt.xlabel("Action")
plt.ylabel("State")
plt.title("Learned Q-table Heatmap")
plt.tight_layout()
plt.show()

## Step 8: Convert the Q-table into a policy

The policy is the action the agent will choose in each state.

For example, if the temperature is cold, the best action may be to slow down.  
If the temperature is hot, the best action may be to speed up.

The learned policy tells us what the Q-table has learned.


In [ ]:
# ============================================================
# Cell 16: Learned policy table
# ============================================================

best_actions = np.argmax(q_table, axis=1)

policy_rows = []

for state in range(N_STATES):
    desc = describe_state(state)
    best_action = int(best_actions[state])

    policy_rows.append({
        "state": state,
        "temp_label": desc["temp_label"],
        "speed_mms": desc["speed_mms"],
        "best_action": best_action,
        "best_action_name": ACTION_NAMES[best_action],
        "Q_slow_down": q_table[state, 0],
        "Q_hold": q_table[state, 1],
        "Q_speed_up": q_table[state, 2],
    })

policy_df = pd.DataFrame(policy_rows)

policy_df

In [ ]:
# ============================================================
# Cell 17: Learned policy grid
# ============================================================

policy_grid = best_actions.reshape(N_TEMP_BINS, N_SPEEDS)

plt.figure(figsize=(9, 5))

plt.imshow(policy_grid, aspect="auto", vmin=0, vmax=2, cmap = "coolwarm")
plt.colorbar(ticks=[0, 1, 2], label="Action: 0 slow, 1 hold, 2 speed up")

plt.xticks(range(N_SPEEDS), [f"{s:.0f}" for s in SPEEDS])
plt.yticks(range(N_TEMP_BINS), TEMP_BIN_NAMES)

for i in range(N_TEMP_BINS):
    for j in range(N_SPEEDS):
        action = int(policy_grid[i, j])
        short_name = {0: "slow", 1: "hold", 2: "up"}[action]
        plt.text(j, i, short_name, ha="center", va="center", fontsize=11)

plt.xlabel("Current speed (mm/s)")
plt.ylabel("Current temperature bin")
plt.title("Learned Policy: Best Action for Each State")
plt.tight_layout()
plt.show()

## Step 9: Test the learned policy

Now we test the learned Q-table in the simulator.

During this test, the agent does not explore randomly.  
It always chooses the action with the highest Q-value for the current state.


In [ ]:
# ============================================================
# Cell 19: Evaluation helper functions
# ============================================================

def q_policy_action(temp_C, speed_index):
    state = state_from_temp_and_speed(temp_C, SPEEDS[speed_index])
    return int(np.argmax(q_table[state]))


def run_policy_episode(policy_function, n_layers=50, initial_temp_C=42.0, initial_speed_mms=70.0, seed=1):
    rng = np.random.default_rng(seed)

    temp = float(initial_temp_C)
    speed_index = speed_to_index(initial_speed_mms)

    rows = []

    for layer in range(1, n_layers + 1):
        state = state_from_temp_and_speed(temp, SPEEDS[speed_index])
        action = policy_function(temp, speed_index)

        next_temp, next_speed_index, reward, next_state = simulator_step(
            temp,
            speed_index,
            action,
            rng=rng
        )

        rows.append({
            "layer": layer,
            "temp_C": temp,
            "speed_mms": SPEEDS[speed_index],
            "state": state,
            "action": action,
            "action_name": ACTION_NAMES[action],
            "next_temp_C": next_temp,
            "next_speed_mms": SPEEDS[next_speed_index],
            "reward": reward,
            "next_state": next_state,
            "in_target_window": TARGET_LO <= next_temp <= TARGET_HI,
        })

        temp = next_temp
        speed_index = next_speed_index

    return pd.DataFrame(rows)


eval_episode = run_policy_episode(q_policy_action, seed=777)

print("Evaluation episode shape:", eval_episode.shape)
print("Layers inside target window:", round(100 * eval_episode["in_target_window"].mean(), 1), "%")
print("Total reward:", round(eval_episode["reward"].sum(), 2))

eval_episode.head()

In [ ]:
# ============================================================
# Cell 20: Plot evaluation temperature
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    eval_episode["layer"],
    eval_episode["next_temp_C"],
    marker="o",
    markersize=4,
    label="Q-policy temperature"
)

plt.axhline(TARGET_LO, linestyle="--", label=f"Target low = {TARGET_LO:.1f}°C")
plt.axhline(TARGET_HI, linestyle="--", label=f"Target high = {TARGET_HI:.1f}°C")

plt.xlabel("Layer")
plt.ylabel("ROI max temperature (°C)")
plt.title("Evaluation Episode Using Learned Q-table")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 21: Plot evaluation speed choices
# ============================================================

plt.figure(figsize=(10, 4))

plt.step(
    eval_episode["layer"],
    eval_episode["next_speed_mms"],
    where="post",
    marker="o",
    markersize=4
)

plt.xlabel("Layer")
plt.ylabel("Print speed (mm/s)")
plt.title("Speed Choices from Learned Q-table")
plt.yticks(SPEEDS)
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 10: Save final outputs

We save the Q-table and supporting result files.

The most important output is:

```text
q_learning_real_outputs/q_table.npy
```

This file contains the learned Q-table.

We also save readable CSV files so that we can inspect the policy and training results later.


In [ ]:
# ============================================================
# Cell 22: Save Q-learning outputs
# ============================================================

from pathlib import Path
import numpy as np

# TODO 22:
# Choose where to save your outputs.
# Recommended: save inside the same Project 4 folder in Google Drive.
# Hint: this line saves inside PROJECT_FOLDER.

OUTPUT_DIR = PROJECT_FOLDER / "q_learning_real_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

q_table_path = OUTPUT_DIR / "q_table.npy"
np.save(q_table_path, q_table)

q_table_readable.to_csv(OUTPUT_DIR / "q_table_readable.csv", index=False)
policy_df.to_csv(OUTPUT_DIR / "learned_policy.csv", index=False)
training_history.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
final_training_episode.to_csv(OUTPUT_DIR / "final_training_episode.csv", index=False)
eval_episode.to_csv(OUTPUT_DIR / "evaluation_episode.csv", index=False)

print("Saved outputs in:", OUTPUT_DIR.resolve())
print("Main Q-table:", q_table_path.resolve())


# Project complete

In this notebook, we trained a Q-learning agent for print speed control.

What we did:

1. Loaded prepared real printer thermal data.
2. Built a simple simulator from layer-to-layer transitions.
3. Defined states, actions, and rewards.
4. Trained a Q-table using Q-learning.
5. Visualized the learned policy.
6. Tested the learned policy in simulation.
7. Saved the final Q-table.
